# 두쫀쿠를 주면 헌혈이 늘어날까? 공공데이터로 직접 검증하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/team-datapopcorn/datapopcorn-assets/blob/main/blood-donation-dzk/blood_gift_analysis.ipynb)

2026년 1월, 헌혈의집에서 '두바이 쫀득 쿠키(두쫀쿠)'를 나눠주자 헌혈자가 줄을 섰다는 뉴스가 쏟아졌습니다. 1월 전국 헌혈은 전년보다 **17.5%** 늘었다고도 보도됐습니다.

이 노트북은 그 숫자를 공공데이터로 다시 계산합니다. 블로그 글 [datapopcorn.ai/blog/blood-donation-gift-dzk-analysis](https://datapopcorn.ai/blog/blood-donation-gift-dzk-analysis)의 모든 숫자와 차트가 이 노트북에서 나옵니다.

**실행 방법**: 메뉴의 `런타임 → 모두 실행`을 누르면 약 1분 안에 끝납니다. GPU는 필요 없습니다. 데이터는 공개 GitHub 저장소에서 바로 읽어 오므로 로그인이나 Drive 연결도 필요 없습니다.

| 데이터 | 출처 | 기간 |
|---|---|---|
| 월별 헌혈 건수 (대한적십자사) | KOSIS 「혈액정보통계」 월별 헌혈통계(DT_445001_002) | 2005.01~2025.12 |
| 2026년 월별 헌혈 건수 | 대한적십자사 혈액관리본부 헌혈통계 페이지 | 2026.01~2026.09 |
| 일별 적혈구제제 보유량 | 혈액관리본부 혈액보유현황 추이 | 매년 11월~4월, 2007~2026 |
| 두쫀쿠 행사 보도 수치 | 연합뉴스·한국경제·동아일보·주간경향 | 2026.01~03 |

In [ ]:
# Step 0. 준비: 설날 계산 패키지와 한글 폰트(Pretendard)를 설치합니다.
!pip -q install korean-lunar-calendar
!mkdir -p fonts && cd fonts && for w in Regular Medium Bold ExtraBold; do [ -f Pretendard-$w.otf ] || wget -q https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/packages/pretendard/dist/public/static/Pretendard-$w.otf; done; ls

In [ ]:
import os, re, io, datetime as dt, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats
from korean_lunar_calendar import KoreanLunarCalendar

# 데이터 위치: 같은 폴더에 data/가 있으면 그걸 쓰고, 없으면 공개 GitHub에서 읽습니다.
DATA_BASE = "data/" if os.path.isdir("data") else "https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/blood-donation-dzk/data/"
RAW_BASE = "raw/" if os.path.isdir("raw") else "https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/blood-donation-dzk/raw/"
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

for f in os.listdir("fonts"):
    if f.endswith(".otf"): font_manager.fontManager.addfont(os.path.join("fonts", f))
plt.rcParams.update({"font.family": "Pretendard", "axes.unicode_minus": False})

# 차트 색: 데이터팝콘 블로그 차트와 같은 팔레트
BG, INK, SUB, GRID = "#F7F6F1", "#1F2328", "#5B636A", "#DEDBD2"
RED, GRAY, BLUE = "#C8415A", "#848D92", "#5B7DB1"
print("DATA_BASE =", DATA_BASE)

In [ ]:
def frame(title, kicker, subtitle, source, size=(12, 7.2)):
    """블로그 차트 공통 틀: 상단 kicker·제목·부제, 하단 출처."""
    fig = plt.figure(figsize=size, dpi=100, facecolor=BG)
    fig.text(0.055, 0.93, kicker, color=RED, fontsize=13, fontweight="bold")
    fig.text(0.055, 0.855, title, color=INK, fontsize=25, fontweight="extra bold")
    fig.text(0.055, 0.80, subtitle, color=SUB, fontsize=13.5)
    fig.add_artist(plt.Line2D([0.055, 0.945], [0.085, 0.085], color=GRID, lw=1.2))
    fig.text(0.055, 0.04, source, color=SUB, fontsize=11)
    ax = fig.add_axes([0.10, 0.17, 0.84, 0.56], facecolor=BG)
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=SUB, labelsize=12, length=0)
    ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
    return fig, ax

def save(fig, name):
    path = f"{OUT}/{name}.png"
    fig.savefig(path, dpi=200, facecolor=BG); plt.show(); print("saved", path)

## Step 1. 질문을 데이터로 바꿉니다

뉴스의 숫자부터 모아 봅니다. 아래 표는 기사에 실린 수치를 그대로 옮긴 것입니다.

여기서 먼저 구분할 것이 있습니다. 기사 속 '2배'는 **행사를 한 헌혈의집에서, 행사 당일에** 센 숫자입니다. 그날 쿠키를 받으러 온 사람 중에는 원래 다음 주에 오려던 사람, 근처 다른 헌혈의집에 가려던 사람도 섞여 있을 수 있습니다. 그러면 그 센터 숫자는 2배가 되어도 **전국 헌혈 총량**은 그만큼 늘지 않습니다.

그래서 질문을 이렇게 바꿉니다.

> 두쫀쿠 행사가 있던 2026년 1~2월, **전국** 헌혈 건수는 평년보다 많았는가?

In [ ]:
events = pd.read_csv(DATA_BASE + "reported_events_2026.csv")
display(events[["start", "end", "where", "what", "reported_numbers"]])

# 숫자로 증가분을 계산할 수 있는 두쫀쿠 행사 3건만 더해 봅니다(엔하이픈 행사는 제외).
reported_increase = {
    "서울중앙 1/16 (561-221)": 561 - 221,
    "광주·전남 1/23 (1,002-356)": 1002 - 356,
    "경남 1/27~29 ((539-264)x3일)": (539 - 264) * 3,
}
for k, v in reported_increase.items(): print(f"{k:32s} +{v:,}건")
REPORTED_SUM = sum(reported_increase.values())
print(f"합계 +{REPORTED_SUM:,}건")

## Step 2. 데이터를 불러오고 정의를 확인합니다

- **헌혈 건수**: 헌혈한 '횟수'입니다. 한 사람이 두 번 하면 2건입니다. 혈액관리본부 페이지는 이를 '헌혈자 수'라고 표시하지만, 실제로는 연인원(건수)입니다.
- **대한적십자사만**: KOSIS 표에는 한마음혈액원 등을 더한 '합계'도 있습니다. 2026년 수치를 제공하는 혈액관리본부 페이지가 대한적십자사 기준이므로 KOSIS에서도 '대한적십자사' 행만 씁니다. (그래서 일부 기사에 나온 '2025년 1월 20만5천 건'은 합계 기준, 이 노트북의 18만8,617건은 적십자 기준입니다.)
- **적혈구제제 보유량**: 그날 0시 기준 재고(Unit)입니다. 헌혈뿐 아니라 병원 사용량에도 좌우됩니다.

두 출처를 이어 붙이기 전에, 겹치는 2024·2025년 값이 서로 똑같은지부터 확인합니다.

In [ ]:
monthly = pd.read_csv(DATA_BASE + "monthly_donations_redcross_2005_2026.csv")
M = monthly.set_index(["year", "month"])["donations"]

# 혈액관리본부 페이지 원본(HTML)에서 '전년 대비 헌혈자 수' 표를 다시 읽어 KOSIS와 대조합니다.
html = urllib.request.urlopen(RAW_BASE + "bloodinfo_bldStat_20261007.html").read().decode("utf-8") \
       if RAW_BASE.startswith("http") else open(RAW_BASE + "bloodinfo_bldStat_20261007.html", encoding="utf-8").read()
seg = html[html.find("전년 대비 헌혈자 수"):]
text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", seg))
page = {}
for mth in range(1, 13):
    m = re.search(rf" {mth}월 ([\d,]+) ([\d,]+) ([\d,]+)", text)
    page[mth] = [int(x.replace(",", "")) for x in m.groups()]   # [2026, 2025, 2024]
check = all(M[(2025, k)] == page[k][1] and M[(2024, k)] == page[k][2] for k in range(1, 13))
print("KOSIS 2024·2025 == 혈액관리본부 페이지 2024·2025 :", check)
assert check
print("2026년 1~9월:", [page[k][0] for k in range(1, 10)])
assert all(M[(2026, k)] == page[k][0] for k in range(1, 10))

stock = pd.read_csv(DATA_BASE + "rbc_stock_daily_winter_2007_2026.csv", parse_dates=["date"]).set_index("date")["rbc_units"]
print(f"월별 {len(monthly)}행 ({monthly.year.min()}~{monthly.year.max()}), 보유량 {len(stock)}일")

## Step 3. 뉴스와 같은 방식으로 먼저 계산해 봅니다

가장 흔한 비교는 '전년 같은 달'과 견주는 것입니다.

In [ ]:
def s(y, months): return sum(M[(y, k)] for k in months)
yoy = lambda y, k: M[(y, k)] / M[(y - 1, k)] - 1
for k in [1, 2, 3]:
    print(f"{k}월: 2025년 {M[(2025,k)]:,} -> 2026년 {M[(2026,k)]:,}  ({yoy(2026,k):+.1%})")
jf = s(2026, [1, 2]) / s(2025, [1, 2]) - 1
print(f"1~2월 합계: {s(2025,[1,2]):,} -> {s(2026,[1,2]):,}  ({jf:+.1%})")

fig, ax = frame("1월 +17.5%, 그런데 2월은 −8.2%",
                "01 / 뉴스가 본 숫자",
                "전국 헌혈 건수(대한적십자사). 두 달을 합치면 증가율은 +4.2%로 줄어듭니다.",
                "출처: KOSIS 월별 헌혈통계, 혈액관리본부 헌혈통계(2026-10-07 조회) · 데이터팝콘 재계산")
xs = np.arange(3); w = 0.36
b25 = [M[(2025, k)] for k in [1, 2, 3]]; b26 = [M[(2026, k)] for k in [1, 2, 3]]
ax.bar(xs - w/2, b25, w, color=GRAY, label="2025년")
ax.bar(xs + w/2, b26, w, color=RED, label="2026년")
for i, k in enumerate([1, 2, 3]):
    ax.text(xs[i] + w/2, b26[i] + 3000, f"{yoy(2026,k):+.1%}", ha="center", color=RED if yoy(2026,k) > 0 else INK,
            fontsize=17, fontweight="bold")
ax.set_xticks(xs, ["1월", "2월", "3월"], fontsize=15, color=INK)
ax.set_ylim(0, 260000); ax.yaxis.set_major_formatter(lambda v, p: f"{v/10000:.0f}만")
ax.legend(frameon=False, fontsize=13, loc="upper right", ncols=2, bbox_to_anchor=(1, 1.13))
save(fig, "01-news-numbers")

## Step 4. 설 연휴 달력을 붙입니다

설 연휴에는 헌혈의집 상당수가 문을 닫고, 단체헌혈도 거의 없습니다. 그래서 **설이 낀 달은 헌혈이 줄어듭니다.**

- 2025년 설날: **1월 29일** → 2025년 1월 헌혈이 적었습니다.
- 2026년 설날: **2월 17일** → 2026년 1월은 연휴가 없는 '꽉 찬 달'이었습니다.

즉 2026년 1월을 2025년 1월과 비교하면, 두쫀쿠가 없었어도 숫자가 크게 늘어날 수밖에 없습니다. 이게 정말인지 21년치로 확인해 봅니다. 설날 날짜는 음력 1월 1일을 양력으로 바꿔 계산합니다.

In [ ]:
cal = KoreanLunarCalendar()
def seollal(y):
    cal.setLunarDate(y, 1, 1, False); return dt.date.fromisoformat(cal.SolarIsoFormat())
SEOLLAL = {y: seollal(y) for y in range(2005, 2027)}
print({y: str(d) for y, d in SEOLLAL.items()})

rows = []
for y in range(2006, 2027):
    prev, cur = SEOLLAL[y - 1].month, SEOLLAL[y].month
    move = "설 1월→2월" if (prev, cur) == (1, 2) else "설 2월→1월" if (prev, cur) == (2, 1) else "같은 달"
    rows.append({"year": y, "jan_yoy": yoy(y, 1), "move": move, "covid": y in (2020, 2021, 2022)})
J = pd.DataFrame(rows).set_index("year")
summary = J[~J.covid & (J.index < 2026)].groupby("move").jan_yoy.agg(["mean", "min", "max", "count"])
display(summary.style.format({"mean": "{:+.1%}", "min": "{:+.1%}", "max": "{:+.1%}"}))
print(f"2026년 1월 전년비 {J.loc[2026,'jan_yoy']:+.1%} (유형: {J.loc[2026,'move']})")

fig, ax = frame("1월 급증은 설이 2월로 옮겨간 해마다 나타납니다",
                "02 / 첫 번째 함정: 설 연휴",
                "1월 헌혈 건수의 전년 대비 증감률. 분홍색은 전년 1월이던 설이 그해 2월로 옮겨간 해, 진한 빨강은 2026년입니다.",
                "출처: KOSIS 월별 헌혈통계(대한적십자사) · 설날은 음력 1월 1일 양력 변환 · 데이터팝콘 재계산")
colors = {"설 1월→2월": "#E7A3B0", "설 2월→1월": BLUE, "같은 달": "#C9CDCF"}
for y, r in J.iterrows():
    c = RED if y == 2026 else colors[r.move]
    ax.bar(y, r.jan_yoy * 100, color=c, width=0.72, alpha=0.45 if r.covid else 1)
ax.axhline(0, color=GRAY, lw=1)
ax.text(2026, J.loc[2026, "jan_yoy"] * 100 + 0.8, "2026\n+17.5%", ha="center", va="bottom", color=RED, fontsize=13, fontweight="bold")
ax.set_xticks(range(2006, 2027, 2)); ax.yaxis.set_major_formatter(lambda v, p: f"{v:+.0f}%")
ax.set_ylim(-17, 23)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=colors[k], label=k) for k in colors] + [Patch(color=GRAY, alpha=0.45, label="코로나(2020~22, 흐리게)")],
          frameon=False, fontsize=12, loc="upper left", ncols=4, bbox_to_anchor=(0, 1.12))
save(fig, "02-seollal-trap")

## Step 5. 1~2월을 합쳐서 설 효과를 지웁니다

설날은 매년 **1월 21일~2월 20일 사이**에 옵니다. 그래서 1월과 2월을 합치면, 설이 어느 달에 있든 연휴는 항상 그 안에 한 번 들어갑니다. 설 효과가 상쇄되는 셈입니다.

다만 2026년에는 헌혈 자체가 줄어드는 흐름도 있었습니다. 그래서 '두쫀쿠 직전' 넉 달(2025년 9~12월)의 전년비도 같이 봅니다.

In [ ]:
before = s(2025, [9, 10, 11, 12]) / s(2024, [9, 10, 11, 12]) - 1
after = s(2026, [3, 4]) / s(2025, [3, 4]) - 1
print(f"직전 흐름  2025년 9~12월 전년비 : {before:+.1%}")
print(f"행사 기간  2026년 1~2월  전년비 : {jf:+.1%}")
print(f"행사 이후  2026년 3~4월  전년비 : {after:+.1%}")
print("설날이 1/21~2/20 안에 있는지:", all(dt.date(y,1,21) <= SEOLLAL[y] <= dt.date(y,2,20) for y in SEOLLAL))

## Step 6. '평년이었다면 몇 건이었을까'를 세 가지 방법으로 추정합니다

사람이 실제로 한 일(두쫀쿠 있는 2026년)과 비교할 '두쫀쿠가 없었다면'의 숫자는 관측할 수 없습니다. 그래서 과거 패턴으로 추정합니다. 이 추정치를 **기준선**이라고 부릅니다.

지표는 간단합니다. **그해 1~2월 헌혈 ÷ 직전 9~12월 헌혈** 비율(R)입니다. 직전 넉 달의 '체력'에 비해 겨울 두 달이 얼마나 나왔는지를 봅니다. 코로나 시기(2020~2022)는 패턴이 깨졌으므로 기준선 계산에서 뺍니다.

기준선을 하나만 고르면 결론이 기준선 선택에 휘둘립니다. 그래서 세 가지를 모두 계산합니다.

In [ ]:
R = pd.Series({y: s(y, [1, 2]) / s(y - 1, [9, 10, 11, 12]) for y in range(2006, 2027)})
train = R.drop([2020, 2021, 2022, 2026])
base = s(2025, [9, 10, 11, 12])

# (1) 전체 평균  (2) 최근 7개 정상 연도 평균  (3) 연도 추세선 + 95% 예측구간
recent = R[[2016, 2017, 2018, 2019, 2023, 2024, 2025]]
x, yv = train.index.values.astype(float), train.values
b, a = np.polyfit(x, yv, 1); resid = yv - (a + b * x); n = len(x)
se = np.sqrt((resid ** 2).sum() / (n - 2))
se_pred = se * np.sqrt(1 + 1 / n + (2026 - x.mean()) ** 2 / ((x - x.mean()) ** 2).sum())
t = stats.t.ppf(0.975, n - 2)
trend = a + b * 2026
baselines = {"① 전체 평균(2006~2025)": train.mean(), "② 최근 7개 정상 연도 평균": recent.mean(), "③ 연도 추세선": trend}
actual = s(2026, [1, 2])
res = pd.DataFrame({k: {"기준선 R": v, "예상 1~2월 건수": round(v * base), "실제 대비": actual / (v * base) - 1}
                    for k, v in baselines.items()}).T
display(res.style.format({"기준선 R": "{:.4f}", "예상 1~2월 건수": "{:,.0f}", "실제 대비": "{:+.1%}"}))
lo, hi = (trend - t * se_pred) * base, (trend + t * se_pred) * base
print(f"실제 2026년 1~2월: {actual:,}건 (R={R[2026]:.4f})")
print(f"추세선 95% 예측구간: {lo:,.0f} ~ {hi:,.0f}건  → 실제값이 구간 {'안' if lo <= actual <= hi else '밖'}에 있습니다")
NOISE_SD = se * base   # 추세선에서 벗어나는 정도(잔차 표준편차)를 건수로 환산
print(f"추세선 기준 평년 흔들림(잔차 표준편차)을 건수로 바꾸면 약 ±{NOISE_SD:,.0f}건")

fig, ax = frame("2026년 1~2월 헌혈은 '평년 범위' 안에 있습니다",
                "03 / 기준선과 비교",
                "R = 그해 1~2월 헌혈 ÷ 직전 9~12월 헌혈. 회색 띠는 추세선 기준 2026년 95% 예측구간입니다.",
                "출처: KOSIS 월별 헌혈통계, 혈액관리본부 헌혈통계 · 코로나(2020~22)는 기준선 계산에서 제외 · 데이터팝콘 재계산")
yrs = np.arange(2006, 2027)
ax.plot(yrs, a + b * yrs, color=GRAY, lw=1.5, ls="--")
ax.fill_between([2025.6, 2026.4], trend - t * se_pred, trend + t * se_pred, color=GRAY, alpha=0.18, lw=0)
for y, v in R.items():
    if y == 2026: continue
    cov = y in (2020, 2021, 2022)
    ax.scatter(y, v, s=70, color=BG if cov else GRAY, edgecolor=GRAY, lw=1.6, zorder=3)
ax.scatter(2026, R[2026], s=150, color=RED, zorder=4)
ax.annotate("2026 (두쫀쿠)", (2026, R[2026]), xytext=(2022.2, 0.505), color=RED, fontsize=14, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=RED, lw=1.2))
ax.text(2006, 0.405, "흰 점: 코로나 시기(기준선 계산 제외)", color=SUB, fontsize=11.5)
ax.set_xticks(range(2006, 2027, 2)); ax.set_ylim(0.40, 0.55)
save(fig, "03-baseline")

## Step 7. 혈액 보유량 반등도 확인합니다

뉴스는 "두쫀쿠 이후 보유량이 3일분에서 5일분 이상으로 회복됐다"고 전했습니다. 그런데 보유량은 **낮을수록 크게 반등하는 성질**이 있습니다. 재고가 바닥이면 혈액원은 단체헌혈 요청, 문자 안내, 각종 기념품 행사를 한꺼번에 쏟아내고, 병원도 사용을 아낍니다. 반대로 재고가 넉넉하게 시작한 해는 겨울에 줄어듭니다.

그래서 해마다 **1월 1일 보유량(출발점)**과 **2월 15일까지의 변화**를 같이 그려 봅니다.

In [ ]:
rows = []
for y in range(2008, 2027):
    j1 = stock.get(pd.Timestamp(f"{y}-01-01"), stock.get(pd.Timestamp(f"{y}-01-02")))
    f15 = stock.get(pd.Timestamp(f"{y}-02-15"))
    rows.append({"year": y, "jan1": j1, "feb15": f15, "chg": f15 / j1 - 1})
ST = pd.DataFrame(rows).set_index("year")
tr = ST.drop(2026)
bs, as_ = np.polyfit(np.log(tr.jan1), np.log1p(tr.chg), 1)
r_val = stats.pearsonr(np.log(tr.jan1), np.log1p(tr.chg))[0]
pred26 = np.expm1(as_ + bs * np.log(ST.loc[2026, "jan1"]))
sd_resid = (np.log1p(tr.chg) - (as_ + bs * np.log(tr.jan1))).std()
print(f"출발점과 변화의 상관계수 r = {r_val:.2f}")
print(f"2026년 1/1 보유량 {ST.loc[2026,'jan1']:,} Unit → 2/15 {ST.loc[2026,'feb15']:,} Unit ({ST.loc[2026,'chg']:+.1%})")
print(f"출발점만으로 예상한 변화 {pred26:+.1%}, 차이는 잔차 표준편차의 {(np.log1p(ST.loc[2026,'chg'])-np.log1p(pred26))/sd_resid:.1f}배")

fig, ax = frame("보유량 반등은 '낮게 시작한 해'의 공통 패턴입니다",
                "04 / 두 번째 함정: 출발점",
                "가로: 1월 1일 적혈구제제 보유량, 세로: 2월 15일까지의 변화율. 선은 2008~2025년 회귀선입니다.",
                "출처: 대한적십자사 혈액관리본부 혈액보유현황 추이(일별, Unit) · 데이터팝콘 재계산")
xx = np.linspace(9000, 50000, 100)
ax.plot(xx / 1000, np.expm1(as_ + bs * np.log(xx)) * 100, color=GRAY, lw=1.5, ls="--")
ax.scatter(tr.jan1 / 1000, tr.chg * 100, s=70, color=GRAY, zorder=3)
for y in [2008, 2016, 2025]:
    ax.annotate(str(y), (tr.loc[y, "jan1"] / 1000, tr.loc[y, "chg"] * 100), xytext=(8, 4), textcoords="offset points",
                color=SUB, fontsize=11)
ax.scatter(ST.loc[2026, "jan1"] / 1000, ST.loc[2026, "chg"] * 100, s=150, color=RED, zorder=4)
ax.annotate("2026", (ST.loc[2026, "jan1"] / 1000, ST.loc[2026, "chg"] * 100), xytext=(10, 6), textcoords="offset points",
            color=RED, fontsize=15, fontweight="bold")
ax.axhline(0, color=GRAY, lw=1)
ax.set_xlabel("1월 1일 보유량 (천 Unit)", color=SUB, fontsize=12)
ax.yaxis.set_major_formatter(lambda v, p: f"{v:+.0f}%"); ax.set_ylim(-70, 170)
save(fig, "04-stock-start")

## Step 8. 규모를 비교합니다: 보도된 증가분은 얼마나 큰가

Step 1에서 더한 '기사에 나온 당일 증가분'을, 전국 1~2월 헌혈 건수, 그리고 평년의 자연스러운 흔들림(Step 6의 표준편차)과 나란히 놓아 봅니다. 데이터가 어떤 크기의 변화까지 구분할 수 있는지를 감으로 잡는 단계입니다.

In [ ]:
print(f"보도된 당일 증가분 합계 : {REPORTED_SUM:,}건 = 1~2월 전국 헌혈의 {REPORTED_SUM/actual:.2%}")
print(f"평년 흔들림(잔차 1 표준편차): ±{NOISE_SD:,.0f}건 = {NOISE_SD/actual:.1%}")

fig, ax = frame("기사 속 증가분은 전국 통계의 '흔들림'보다 작습니다",
                "05 / 규모 비교",
                "보도된 두쫀쿠 당일 증가분(서울중앙·광주전남·경남) 합계와 평년 흔들림을 같은 눈금에 놓았습니다.",
                "출처: 연합뉴스(1/23), 동아일보(1/26), 한국경제(1/31), KOSIS · 데이터팝콘 재계산")
labels = ["보도된 당일 증가분 합계", "평년 흔들림 (잔차 1 표준편차)", "2026년 1~2월 전국 헌혈"]
vals = [REPORTED_SUM, NOISE_SD, actual]
cols = [RED, GRAY, "#C9CDCF"]
ax.barh(range(3)[::-1], vals, color=cols, height=0.55)
for i, v in zip(range(3)[::-1], vals):
    ax.text(v + 4000, i, f"{v:,.0f}건", va="center", fontsize=16, fontweight="bold", color=INK)
ax.set_yticks(range(3)[::-1], labels, fontsize=14, color=INK)
ax.set_xlim(0, 480000); ax.xaxis.set_major_formatter(lambda v, p: f"{v/10000:.0f}만")
ax.grid(axis="y", visible=False); ax.grid(axis="x", color=GRID, lw=0.8)
fig.axes[0].set_position([0.29, 0.17, 0.65, 0.56])
save(fig, "05-scale")

## Step 9. 결론과 한계

1. **센터 단위 효과는 사실입니다.** 행사 당일 해당 헌혈의집 헌혈은 2~3배가 됐습니다(보도 수치).
2. **1월 +17.5%의 대부분은 설 연휴 이동입니다.** 설이 1월에서 2월로 옮겨간 해의 1월은 평균적으로 크게 늡니다.
3. **1~2월을 합치면 +4.2%이고, 기준선 세 가지 모두 '평년 범위' 안입니다.** 기준선에 따라 −1.4%~+5.8%이며, 추세선의 95% 예측구간 안에 들어갑니다.
4. **보유량 반등도 출발점이 낮았던 해의 공통 패턴과 비슷합니다.**

이 결과는 "두쫀쿠가 효과가 없었다"는 뜻이 **아닙니다.** 효과가 있었더라도 그 크기가 전국 월별 통계의 자연스러운 흔들림보다 작아서, **공개된 데이터로는 구분할 수 없다**는 뜻입니다. 또 같은 기간 엔하이픈 포토카드, 기념품 1+1, 혈액 부족 보도 같은 다른 자극이 함께 있었기 때문에, 설령 초과분이 보였더라도 두쫀쿠 하나의 몫으로 나눌 수 없습니다.

제대로 답하려면 **헌혈의집별·일별 헌혈 건수**와 **행사 일정**이 필요합니다. 그러면 행사를 한 센터와 안 한 센터를, 행사 전후로 비교하는 이중차분(DiD) 분석을 할 수 있습니다. 현재 이 자료는 공개되지 않습니다.

## (선택) 데이터를 직접 다시 수집하기

위 분석은 저장소에 보관한 스냅샷을 씁니다. 원본에서 직접 다시 받고 싶다면 아래 셀의 `RUN_COLLECT`를 `True`로 바꾸세요.

- 혈액보유량: 혈액관리본부 페이지가 2개월 단위로 일별 값을 돌려줍니다. 해외 서버(Colab)에서는 응답이 느리거나 막힐 수 있습니다.
- KOSIS 월별 헌혈통계: [통계표 링크](https://kosis.kr/statHtml/statHtml.do?orgId=445&tblId=DT_445001_002)에서 `시점`으로 2005~2025년을 모두 고른 뒤 CSV로 내려받습니다. 칸이 많아 화면 대신 다운로드 창이 바로 뜹니다.

In [ ]:
RUN_COLLECT = False

def fetch_stock(end):
    """end 날짜로 끝나는 2개월치 일별 적혈구제제 보유량을 받아 (날짜, Unit) 목록으로 돌려줍니다."""
    url = "https://www.bloodinfo.net/knrcbs/bi/info/bldHoldSttus.do?mi=1046"
    data = f"selRbcYear={end.year}&selRbcMon={end.month:02d}&selRbcDay={end.day:02d}&selRbcTerm=2".encode()
    h = urllib.request.urlopen(urllib.request.Request(url, data=data, headers={"User-Agent": "Mozilla/5.0"}), timeout=60).read().decode()
    rows = re.findall(r"getData = (\d+);\s*dataRow = \[getData.toString\(\), (\d+)\]", h.split("function drawPlt")[0])
    out, cur = [], end
    for d, v in reversed(rows):           # 그래프는 '일'만 주므로 끝 날짜부터 거꾸로 날짜를 맞춥니다(결측일 건너뜀).
        while cur.day != int(d): cur -= dt.timedelta(days=1)
        out.append((cur, int(v))); cur -= dt.timedelta(days=1)
    return sorted(out)

if RUN_COLLECT:
    sample = fetch_stock(dt.date(2026, 2, 28))
    print(len(sample), sample[:2], sample[-2:])
    print("저장소 값과 일치:", all(stock[pd.Timestamp(d)] == v for d, v in sample))